In [1]:

from pathlib import Path
import numpy as np
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from chemprop.featurizers.atom import MultiHotAtomFeaturizer
from rdkit.Chem.rdchem import HybridizationType
import pandas as pd
from rdkit import Chem
from chemprop import data, featurizers, models, nn
from ml_enhance import QuantumFPFileLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from typing import Iterable
import pandas as pd
from chemprop.data import MoleculeDatapoint, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer

In [ ]:
# check whether the order of the smiles in my used DF matches the order of the files list
check = []
for df_id, file in zip(used_ids, used_files, strict=True):
    matches = re.findall(r"\d+", file.name)
    id = int(matches[1])

    check.append(id == df_id)

np.sum(check) # is correct

In [ ]:
df.query("id == 5")["solubility"].iloc[0]

In [2]:
from data_preprocess import filter_files

In [ ]:
df = pd.read_csv("../data/processed_dataset_rerun.csv")
used_ids: list[int] = df["id"].apply(round).to_list()

qfp_loader = QuantumFPFileLoader(Path("../data/QFP_rerun/output"))
filelist: list[Path] = qfp_loader.list_output_files()
used_files = np.array(filter_files(filelist, used_ids))

outer_splits = pd.read_pickle("../hpc_splits.pkl")
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

config = {
    "use_atom_features": False,
    "use_bond_features": False,
    "use_mol_features": False,
    "use_custom_atom_featurizer": False,
    "use_custom_bond_featurizer": False,
    "n_rbf_centers": 10,
    "target_col": "solubility",
}

In [ ]:
tr_idxs, tst_idxs = outer_splits[3]
outer_train_files = used_files[tr_idxs][:2]
outer_test_files = used_files[tst_idxs][:2]

In [ ]:
df.filter(regex="atomic").columns

In [ ]:
# outer_train_dataset, outer_test_dataset, outer_target_scaler = build_fold(
#     outer_train_files, outer_test_files, df, qfp_loader, **config
# )

In [ ]:
for sdf in qfp_loader.stream_conformer_dataframe(used_files[3]):
    tdf = sdf

tdf.loc[0, ["original_smiles", "output_smiles"]].values
tdf.loc[6, ["original_smiles", "output_smiles"]].values

# np.array(tdf["partial_charge"].values.tolist())[:, :, 1]
# tdf[["original_smiles", "partial_charge"]].head(5)
# tdf["partial_charge"][0]
# tdf["original_smiles"][0]

In [ ]:
'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

In [ ]:
'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O-:12])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

In [ ]:
'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

In [ ]:
train_features = process_files(
    outer_train_files,
    qfp_loader,
    use_atom_features=True,
    use_bond_features=False,
    use_mol_features=False,
)
val_features = process_files(
    outer_test_files,
    qfp_loader,
    use_atom_features=True,
    use_bond_features=False,
    use_mol_features=False,
)

In [ ]:
train_atom_df = train_features["atoms"]
val_atom_df = val_features["atoms"]

In [4]:
atomic_features: list[str] = [
    "atomic_fukui_minus",
    "atomic_fukui_plus",
    "atomic_dipole_norm",
    "atomic_quadrupole_principal_invariant_2",
    "atomic_quadrupole_principal_invariant_3",
    "atomic_polarizability_mean",
    "atomic_polarizability_anisotropy",
    "atomic_sasa",
    "partial_charge",
]

In [ ]:
n_rbf_centers = 10

train_atom_df, val_atom_df, _ = scale_features(train_atom_df, val_atom_df, atomic_features)
train_atom_df.head()

In [ ]:
train_atom_df, atom_rbf_cols = apply_rbf(train_atom_df, atomic_features, n_rbf_centers)
# val_atom_df, _ = apply_rbf(val_atom_df, atomic_features, n_rbf_centers)

In [ ]:
train_atom_df.head()

In [5]:
from collections.abc import Generator

def stream_conformer_df(
    file: Path,
    loader: QuantumFPFileLoader,
) -> Generator[pd.DataFrame, None, None]:
    for sdf in loader.stream_conformer_dataframe(file):
        sdf["gibbs_free_energy_300K"] = sdf["gibbs_free_energy"].map(lambda x: x[1][1])
        yield sdf

In [29]:
for sdf in stream_conformer_df(used_files[120], qfp_loader):
    tdf = sdf[["original_smiles", "gibbs_free_energy_300K", *atomic_features]]
    atoms_per_conformer = tdf["atomic_dipole_norm"].apply(lambda x: len(x))
    display(atoms_per_conformer)

0    17
1    17
Name: atomic_dipole_norm, dtype: int64

In [ ]:
def get_info(file):
    for sdf in stream_conformer_df(file, qfp_loader):
        tdf = sdf[["original_smiles", "gibbs_free_energy_300K", *atomic_features]]
        atoms_per_conformer = tdf["atomic_dipole_norm"].apply(len)

        n_unique_sizes = atoms_per_conformer.nunique()
        
        energies = tdf["gibbs_free_energy_300K"]

        return {
            "original_smiles": tdf["original_smiles"].iloc[0],
            "id": sdf["id"].iloc[0],
            "n_conformers": len(atoms_per_conformer),
            "atom_counts": atoms_per_conformer.unique().tolist(),
            "energies": energies.to_numpy(),
            "inconsistent": n_unique_sizes > 1,
        }

In [19]:
from ml_enhance import parallelize

results = parallelize(get_info, used_files[:100], n_jobs=6)

100%|██████████| 100/100 [00:25<00:00,  3.94it/s]


In [20]:
result_df = pd.DataFrame(results)
result_df.head(10)

,original_smiles,id,n_conformers,atom_counts,energies,inconsistent
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,0,1,[20],"0 30.821674 Name: gibbs_free_energy_300K, d...",False
1,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,1,1,[14],"0 7.998828 Name: gibbs_free_energy_300K, dt...",False
2,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,10,20,[33],0 93.144375 1 92.637493 2 93.13233...,False
3,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,100,4,[25],0 49.611325 1 50.116866 2 50.932086 3...,False
4,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,1000,10,[34],0 21.425637 1 21.604573 2 21.553099 3...,False
5,[C:1](/[C:2](=[C:3](\[C:4]([H:22])([H:23])[H:2...,1001,31,[43],0 136.014838 1 136.065359 2 138.24...,False
6,[C:1]([C:2](=[C:3]([H:31])[H:32])[C:4](=[O:5])...,1002,32,[36],0 -23.339723 1 -23.504020 2 -24.08997...,False
7,[O:1]=[C:2]1[C:3]([H:28])=[C:4]([H:29])[C:5](=...,1003,5,[41],0 66.872309 1 68.933571 2 65.024632 3...,False
8,[C:1]([C:2]([C:3]([C:4]([C:5]([C:6]([C:7]([C:8...,1004,32,[45],0 169.805945 1 169.065043 2 168.36...,False
9,[C:1]([C:2]([C:3]([H:37])([H:38])[H:39])([c:4]...,1005,26,[65],0 179.272612 1 179.212795 2 179.22...,False


In [ ]:
result_df["lens"] = result_df["atom_counts"].apply(len)

result_df.query("inconsistent == True").sort_values('lens', ascending=False).head(20)

In [ ]:
result_df.iloc[1898].original_smiles

In [ ]:
'[O:1]([c:2]1[c:3]([H:21])[c:4]([O:5][H:22])[c:6]2[c:17]([c:18]1[O:19][H:27])[C:15](=[O:16])[c:14]1[c:9]([c:10]([H:23])[c:11]([H:24])[c:12]([H:25])[c:13]1[H:26])[C:7]2=[O:8])[H:20]'

In [ ]:
for sdf in stream_conformer_df(used_files[0], qfp_loader):
    print(sdf)